# Full `ADF conversion` Prefect Flow
  
Thprefect flow is doing the following:



## Initialisation

In [ ]:
# Imports
import ast
from IPython.display import JSON
import os
import os.path as osp

from resources.widget_utils import *

from rs_client.ogcapi.dpr_client import DprProcessor
from rs_common.prefect_utils import *
from rs_workflows.auxip_flow import auxip_staging
from rs_workflows.flow_utils import  DprProcessIn, Priority, ProcessingMode, WorkflowType
from rs_workflows.init_pi_db_flow import init_pi_database
from rs_workflows.adf_flow import adf_conversion

In [ ]:
print(f"Prefect server URL used internally: {os.environ['PREFECT_API_URL']}")
dashboard_url = f"{os.environ['RSPY_PREFECT_URL']}/dashboard"
print(f"Prefect dashboard public URL: {dashboard_url}")

In [ ]:
# Choose prefect deployment method
deploy_prefect_radio

In [ ]:
# Choose prefect flow run method
run_prefect_radio

In [ ]:
# Choose adf type to convert
adf_proc_radio

In [ ]:
# Init environment before running a demo notebook.
from resources.utils import *  

init_demo()
await init_dask_cluster_staging(scale=2)


# Reload the global vars again
from resources.utils import *

In [ ]:
# Create test collections
AUXIP_COLLECTION = "TEST_FLOW_AUXIP"
        
create_test_collection(AUXIP_COLLECTION)
  
# Prefect flow environment arguments
flow_env_args = {
  "env": {
    "owner_id": OWNER_ID,
  },
}

# DPR processing input parameters
adg_process_in = AdfProcessIn(
    **flow_env_args,         
    adf_type=adf_proc_radio.value,     
    auxiliary_product_to_collection_identifier = [{"product_type": "*", "collection_name": AUXIP_COLLECTION}],    
    start_datetime="2014-01-01T11:00:00Z",
    end_datetime="2025-10-03T11:00:00Z",
    satellite=None,
)

## Deploy rs-client-libraries Prefect flows

In [ ]:
# Deploy the Prefect flows
adf_conversion_deploy, auxip_staging_deploy = await deploy_prefect(
    deploy_file="./adf_conversion_flow.yaml", 
    s3_code_folder=s3_code_folder, 
    work_pool_name=os.environ["PREFECT_WORK_POOL_EOPF"]
)

## Run the ADF conversion flow

In [ ]:
print(f"Run demo for: {dpr_proc_radio.value!r}")

# Run the processor
params = {"adf_input": adf_conversion_in.model_dump(mode="json")}
state = await run_prefect(
    deploy_name=adf_conversion_deploy, 
    py_func=adf_conversion, 
    params=params
)
if state is not None:
    flow_run_id = state.state_details.flow_run_id
    print(f"Flow run id: {flow_run_id!r}")
    if state.is_failed() or state.is_crashed():
        # Retrieve logs of failed flow run
        response = http_session.get(f"{os.environ['PREFECT_API_URL']}/flow_runs/{flow_run_id}/logs/download")
        response.raise_for_status()
        logs_text = response.text

        print("=== Prefect flow logs ===")
        print(logs_text)
        print("=== End of logs ===")

        raise RuntimeError(f"Prefect flow {flow_run_id} failed.\n\nLogs:\n{logs_text}")